#**Probability**

Suppose you're playing a card game with a standard deck of 52 cards. You decide to only focus on the face cards—which are the Jack, Queen, and King from each of the four suits (Hearts, Diamonds, Clubs, Spades). So in total, there are 12 face cards (3 cards × 4 suits).

Now, you want to find the probability that if you randomly pick one face card, it turns out to be a King. There are 4 Kings among the 12 face cards, so theoretical probability is 4/12 = 0.333.

To verify this using code, you run a simulation 100,000 times, where in each trial, you randomly pick a card from the face cards. Every time it's a King, you count it. At the end, you divide the number of Kings you got by the total number of trials to estimate the simulated probability.

In [ ]:
import random

# Create a deck of cards
suits = ['Hearts', 'Diamonds', 'Clubs', 'Spades']
ranks = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'Jack', 'Queen', 'King', 'Ace']
deck = [(rank, suit) for suit in suits for rank in ranks]
print(deck)
# Filter face cards (Jack, Queen, King)
face_cards = [card for card in deck if card[0] in ['Jack', 'Queen', 'King']]

# Simulation parameters
trials = 100000
king_given_face = 0

# Run simulation
for _ in range(trials):
    card = random.choice(face_cards)
    if card[0] == 'King':
        king_given_face += 1

# Calculate simulated conditional probability
simulated_probability = king_given_face / trials

# Theoretical probability
theoretical_probability = 4 / 12  # 4 Kings out of 12 face cards

print(f"Theoretical P(King | Face card): {theoretical_probability:.3f}")
print(f"Simulated P(King | Face card):   {simulated_probability:.3f}")

[('2', 'Hearts'), ('3', 'Hearts'), ('4', 'Hearts'), ('5', 'Hearts'), ('6', 'Hearts'), ('7', 'Hearts'), ('8', 'Hearts'), ('9', 'Hearts'), ('10', 'Hearts'), ('Jack', 'Hearts'), ('Queen', 'Hearts'), ('King', 'Hearts'), ('Ace', 'Hearts'), ('2', 'Diamonds'), ('3', 'Diamonds'), ('4', 'Diamonds'), ('5', 'Diamonds'), ('6', 'Diamonds'), ('7', 'Diamonds'), ('8', 'Diamonds'), ('9', 'Diamonds'), ('10', 'Diamonds'), ('Jack', 'Diamonds'), ('Queen', 'Diamonds'), ('King', 'Diamonds'), ('Ace', 'Diamonds'), ('2', 'Clubs'), ('3', 'Clubs'), ('4', 'Clubs'), ('5', 'Clubs'), ('6', 'Clubs'), ('7', 'Clubs'), ('8', 'Clubs'), ('9', 'Clubs'), ('10', 'Clubs'), ('Jack', 'Clubs'), ('Queen', 'Clubs'), ('King', 'Clubs'), ('Ace', 'Clubs'), ('2', 'Spades'), ('3', 'Spades'), ('4', 'Spades'), ('5', 'Spades'), ('6', 'Spades'), ('7', 'Spades'), ('8', 'Spades'), ('9', 'Spades'), ('10', 'Spades'), ('Jack', 'Spades'), ('Queen', 'Spades'), ('King', 'Spades'), ('Ace', 'Spades')]
Theoretical P(King | Face card): 0.333
Simulated P

In [ ]:
pip install pgmpy


#**Bayesian Network**

**Scenario:** Monty Hall Problem using Bayesian Network

Imagine a game show like the Monty Hall problem:

There are three doors: behind one is a car (prize), behind the other two are goats.

A guest picks one door (say door 1).

The host Monty knows where the prize is, and opens one of the other two doors with a goat behind it.

Now, the guest can either stick with their choice or switch to the other unopened door.

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Create the Bayesian Network structure
model = DiscreteBayesianNetwork([
    ('Guest', 'Monty'),    # Guest's choice affects Monty's choice
    ('Prize', 'Monty')     # Prize location affects Monty's choice
])

# Define CPD for Guest's choice (uniform distribution)
cpd_guest = TabularCPD(
    variable='Guest',
    variable_card=3,
    values=[[1/3], [1/3], [1/3]],
    state_names={'Guest': [1, 2, 3]}
)

# Define CPD for Prize location (uniform distribution)
cpd_prize = TabularCPD(
    variable='Prize',
    variable_card=3,
    values=[[1/3], [1/3], [1/3]],
    state_names={'Prize': [1, 2, 3]}
)

# Define CPD for Monty's choice given Guest and Prize
cpd_monty = TabularCPD(
    variable='Monty',
    variable_card=3,
    values=[
        # G=1 P=1  G=1 P=2  G=1 P=3  G=2 P=1  G=2 P=2  G=2 P=3  G=3 P=1  G=3 P=2  G=3 P=3
        [0,      0,      0,      0,      1,      0.5,    0,      0.5,    1],  # Monty=1
        [0.5,    0,      1,      0,      0,      0,      1,      0.5,    0],  # Monty=2
        [0.5,    1,      0,      1,      0,      0.5,    0,      0,      0]   # Monty=3
    ],
    evidence=['Guest', 'Prize'],
    evidence_card=[3, 3],
    state_names={
        'Monty': [1, 2, 3],
        'Guest': [1, 2, 3],
        'Prize': [1, 2, 3]
    }
)

# Add CPDs to the model
model.add_cpds(cpd_guest, cpd_prize, cpd_monty)

# Check if the model is valid
assert model.check_model()

# Perform inference
infer = VariableElimination(model)

# Scenario: Guest picks door 1, Monty opens door 3
# What's the probability of prize being behind door 2?
result = infer.query(
    variables=['Prize'],
    evidence={'Guest': 1, 'Monty': 3}
)

print("Probability distribution of the prize location given Guest=1 and Monty=3:")
print(result)

# Probability of winning if sticking with original choice (door 1)
print("\nProbability of winning if sticking with door 1:")
print(result.values[0])  # Index 0 corresponds to Prize=1

# Probability of winning if switching to door 2
print("Probability of winning if switching to door 2:")
print(result.values[1])  # Index 1 corresponds to Prize=2

Probability distribution of the prize location given Guest=1 and Monty=3:
+----------+--------------+
| Prize    |   phi(Prize) |
+==========+==============+
| Prize(1) |       0.3333 |
+----------+--------------+
| Prize(2) |       0.6667 |
+----------+--------------+
| Prize(3) |       0.0000 |
+----------+--------------+

Probability of winning if sticking with door 1:
0.3333333333333333
Probability of winning if switching to door 2:
0.6666666666666666


#🔔 **Scenario: Burglary & Alarm System**
Imagine a smart home with a security alarm system. Here’s what’s going on:

Burglary and Earthquake are two independent events that can trigger the Alarm.

If the Alarm goes off, it can lead to phone calls from two neighbors: David and Sophia.

David and Sophia may or may not call, depending on whether they hear the alarm.

So the network models the probability of a burglary or earthquake occurring, and how that affects the alarm and whether neighbors call.

**TabularCPD**	Used to define the probability distribution for a variable

**variable**	The name of the variable

**variable_card**	Number of possible states the variable can take (2 = True/False)

**values**	Probability table (rows = variable states, columns = parent combinations)

**evidence**	Parent variables (if any)

**evidence_card**	Number of states each parent has

**state_names**	Human-readable names for each stat

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Define the Bayesian Network structure
model = DiscreteBayesianNetwork([
    ('Burglary', 'Alarm'),
    ('Earthquake', 'Alarm'),
    ('Alarm', 'David_calls'),
    ('Alarm', 'Sophia_calls')
])

# CPD for Burglary
cpd_burglary = TabularCPD(
    variable='Burglary',
    variable_card=2,
    values=[[0.999], [0.001]],  # [False, True]
    state_names={'Burglary': ['False', 'True']}
)

# CPD for Earthquake
cpd_earthquake = TabularCPD(
    variable='Earthquake',
    variable_card=2,
    values=[[0.998], [0.002]],  # [False, True]
    state_names={'Earthquake': ['False', 'True']}
)

# CPD for Alarm (depends on Burglary and Earthquake)
cpd_alarm = TabularCPD(
    variable='Alarm',
    variable_card=2,
    values=[
        # B=F E=F  B=F E=T  B=T E=F  B=T E=T
        [0.999,   0.71,   0.06,   0.05],  # Alarm=False
        [0.001,   0.29,   0.94,   0.95]   # Alarm=True
    ],
    evidence=['Burglary', 'Earthquake'],
    evidence_card=[2, 2],
    state_names={
        'Alarm': ['False', 'True'],
        'Burglary': ['False', 'True'],
        'Earthquake': ['False', 'True']
    }
)

# CPD for David_calls (depends on Alarm)
cpd_david = TabularCPD(
    variable='David_calls',
    variable_card=2,
    values=[
        # A=F    A=T
        [0.95,  0.1],  # David_calls=False
        [0.05,  0.9]   # David_calls=True
    ],
    evidence=['Alarm'],
    evidence_card=[2],
    state_names={
        'David_calls': ['False', 'True'],
        'Alarm': ['False', 'True']
    }
)

# CPD for Sophia_calls (depends on Alarm)
cpd_sophia = TabularCPD(
    variable='Sophia_calls',
    variable_card=2,
    values=[
        # A=F    A=T
        [0.99,  0.3],  # Sophia_calls=False
        [0.01,  0.7]   # Sophia_calls=True
    ],
    evidence=['Alarm'],
    evidence_card=[2],
    state_names={
        'Sophia_calls': ['False', 'True'],
        'Alarm': ['False', 'True']
    }
)

# Add CPDs to the model
model.add_cpds(cpd_burglary, cpd_earthquake, cpd_alarm, cpd_david, cpd_sophia)

# Check if the model is valid
assert model.check_model()

# Perform inference
infer = VariableElimination(model)

# Calculate P(Alarm=True | Burglary=True)
prob_alarm_given_b_true = infer.query(
    variables=['Alarm'],
    evidence={'Burglary': 'True'}
)
print("P(Alarm=True | Burglary=True):", prob_alarm_given_b_true.values[1])  # Index 1 is 'True'

# Calculate P(Alarm=True | Burglary=False)
prob_alarm_given_b_false = infer.query(
    variables=['Alarm'],
    evidence={'Burglary': 'False'}
)
print("P(Alarm=True | Burglary=False):", prob_alarm_given_b_false.values[1])  # Index 1 is 'True'

# Optional: Calculate P(Burglary=True | Alarm=True)
prob_burglary_given_a_true = infer.query(
    variables=['Burglary'],
    evidence={'Alarm': 'True'}
)
print("P(Burglary=True | Alarm=True):", prob_burglary_given_a_true.values[1])  # Index 1 is 'True'

P(Alarm=True | Burglary=True): 0.94002
P(Alarm=True | Burglary=False): 0.0015780000000000002
P(Burglary=True | Alarm=True): 0.373551228281836


#**=============== HIDDEN MARKOV MODEL ==================**

#🎯 **Scenario: Color-Changing Magic Ball**
Imagine you have a magic ball that can only be either Red or Blue. Every second, it randomly changes color.

Here’s the twist:

If the ball is Red, it has a 50% chance to stay Red and a 50% chance to turn Blue.

If the ball is Blue, same thing — 50% chance to stay Blue, 50% chance to turn Red.

This setup is an example of a Markov Process, where:

The next state depends only on the current state (not on the past history).

It's a memoryless process.



In [ ]:
import numpy as np

# Define the states and transition matrix
states = ["Red", "Blue"]
transition_matrix = np.array([[0.5, 0.5],  # From Red -> Red or Blue
                              [0.5, 0.5]]) # From Blue -> Red or Blue

# Function to simulate the Markov process
def simulate_markov_process(initial_state, num_steps):
    current_state = initial_state
    state_sequence = [current_state]

    for _ in range(num_steps):
        if current_state == "Red":
            next_state = np.random.choice(states, p=transition_matrix[0])
        else:
            next_state = np.random.choice(states, p=transition_matrix[1])

        state_sequence.append(next_state)
        current_state = next_state

    return state_sequence

# Simulate the process starting from "Red" and for 10 steps
initial_state = "Red"
num_steps = 10
state_sequence = simulate_markov_process(initial_state, num_steps)

# Output the sequence of states
print(f"State sequence for {num_steps} steps starting from {initial_state}:")
print(" -> ".join(state_sequence))


State sequence for 10 steps starting from Red:
Red -> Blue -> Red -> Blue -> Red -> Red -> Blue -> Blue -> Red -> Blue -> Red


In [2]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 39.5 MB/s eta 0:00:00


In [3]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
# Step 1: Define the structure of the Bayesian Network
model = DiscreteBayesianNetwork([
('Burglary', 'Alarm'),
('Earthquake', 'Alarm'),
('Alarm', 'JohnCalls'),
('Alarm', 'MaryCalls')
])
# Step 2: Define the CPDs (Conditional Probability Distributions)
# P(Burglary)
cpd_burglary = TabularCPD(variable='Burglary', variable_card=2, values=[[0.999], [0.001]])
# P(Earthquake)
cpd_earthquake = TabularCPD(variable='Earthquake', variable_card=2, values=[[0.998], [0.002]])
# P(Alarm | Burglary, Earthquake)
cpd_alarm = TabularCPD(
variable='Alarm',
variable_card=2,

values=[
[0.999, 0.71, 0.06, 0.05], # Alarm = False
[0.001, 0.29, 0.94, 0.95] # Alarm = True
],
evidence=['Burglary', 'Earthquake'],
evidence_card=[2, 2]
)
# P(JohnCalls | Alarm)
cpd_john = TabularCPD(
variable='JohnCalls',
variable_card=2,
values=[
[0.3, 0.9], # JohnCalls = False
[0.7, 0.1] # JohnCalls = True
],
evidence=['Alarm'],
evidence_card=[2]
)
# P(MaryCalls | Alarm)
cpd_mary = TabularCPD(
variable='MaryCalls',
variable_card=2,
values=[
[0.2, 0.99], # MaryCalls = False
[0.8, 0.01] # MaryCalls = True
],
evidence=['Alarm'],
evidence_card=[2]
)
# Step 3: Add CPDs to the model
model.add_cpds(cpd_burglary, cpd_earthquake, cpd_alarm, cpd_john, cpd_mary)
# Step 4: Verify the model
assert model.check_model(), "Model is incorrect"
# Step 5: Perform inference
inference = VariableElimination(model)
# Query: What is the probability of a burglary given that both John and Mary called?
result = inference.query(variables=['Burglary'], evidence={'JohnCalls': 1, 'MaryCalls': 1})
print(result)


+-------------+-----------------+
| Burglary    |   phi(Burglary) |
+=============+=================+
| Burglary(0) |          0.9999 |
+-------------+-----------------+
| Burglary(1) |          0.0001 |
+-------------+-----------------+
